In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import evaluate, evaluate_datasets

ROOT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

DATA_ROOT = (
    ROOT
    / "data/raw/competition"
    / "biohub-cell-tracking-during-development"
)

TRAIN_DIR = DATA_ROOT / "train"

train_zarrs = sorted(
    TRAIN_DIR.glob("*.zarr")
)

len(train_zarrs)

In [ ]:
sanity_stem = None

for zarr_path in train_zarrs:

    stem = zarr_path.stem

    ds = open_dataset(
        TRAIN_DIR / stem,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    node_ids = ds.tracks.node_ids()

    out_degree = np.asarray(
        ds.tracks.out_degree(node_ids)
    )

    if np.any(out_degree == 2):
        sanity_stem = stem
        break

print("Sanity dataset:", sanity_stem)

In [ ]:
pred_ds = open_dataset(
    TRAIN_DIR / sanity_stem,
    normalize=False,
    require_tracks=True,
    load_image=False,
)

gt_ds = open_dataset(
    TRAIN_DIR / sanity_stem,
    normalize=False,
    require_tracks=True,
    load_image=False,
)

pred_graph = pred_ds.tracks
gt_graph = gt_ds.tracks

scale = tuple(
    float(v)
    for v in gt_ds.scale
)

print("scale:", scale)
print("GT nodes:", gt_graph.num_nodes())
print("GT edges:", gt_graph.num_edges())

gt_node_ids = gt_graph.node_ids()

n_gt_divisions = int(
    np.sum(
        np.asarray(
            gt_graph.out_degree(
                gt_node_ids
            )
        ) == 2
    )
)

print(
    "GT divisions:",
    n_gt_divisions,
)

In [ ]:
from tracking_cellmot.metrics import (
    evaluate,
)

result = evaluate(
    pred_graph,
    gt_graph,
    scale=scale,
    max_distance=7.0,
)

result

In [ ]:
edge_denom = (
    result.edge_tp
    + result.edge_fp
    + result.edge_fn
)

edge_jaccard = (
    result.edge_tp
    / edge_denom
)

division_denom = (
    result.division_tp
    + result.division_fp
    + result.division_fn
)

division_jaccard = (
    result.division_tp
    / division_denom
)

raw_score = (
    edge_jaccard
    + 0.1 * division_jaccard
)

print(
    "Edge TP / FP / FN:",
    result.edge_tp,
    result.edge_fp,
    result.edge_fn,
)

print(
    "Division TP / FP / FN:",
    result.division_tp,
    result.division_fp,
    result.division_fn,
)

print(
    "Edge Jaccard:",
    edge_jaccard,
)

print(
    "Division Jaccard:",
    division_jaccard,
)

print(
    "Raw combined score:",
    raw_score,
)

In [ ]:
import tracksdata as td

PHYSICAL_SCALE = (
    1.625,
    0.40625,
    0.40625,
)


def make_two_node_track(x_shift_um=0.0):
    """
    Create:
        t=0 node -> t=1 node

    Both nodes are shifted by x_shift_um
    relative to the origin.
    """

    graph = td.graph.InMemoryGraph()

    graph.add_node_attr_key(
        "z",
        pl.Float64,
        0.0,
    )

    graph.add_node_attr_key(
        "y",
        pl.Float64,
        0.0,
    )

    graph.add_node_attr_key(
        "x",
        pl.Float64,
        0.0,
    )

    # Convert physical µm shift into x voxel coordinates
    x_voxel_shift = (
        x_shift_um
        / PHYSICAL_SCALE[2]
    )

    n0 = graph.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0 + x_voxel_shift,
        }
    )

    n1 = graph.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0 + x_voxel_shift,
        }
    )

    graph.add_edge(
        n0,
        n1,
        {},
    )

    return graph

In [ ]:
gt_synth = make_two_node_track(
    x_shift_um=0.0
)

pred_synth = make_two_node_track(
    x_shift_um=0.0
)

evaluate(
    pred_synth,
    gt_synth,
    scale=PHYSICAL_SCALE,
    max_distance=7.0,
)

In [ ]:
boundary_records = []

test_distances_um = [
    0.0,
    6.0,
    6.9,
    6.99,
    7.0,
    7.01,
    7.1,
    8.0,
]

for distance_um in test_distances_um:

    gt_graph = make_two_node_track(
        x_shift_um=0.0
    )

    pred_graph = make_two_node_track(
        x_shift_um=distance_um
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    denom = (
        result.edge_tp
        + result.edge_fp
        + result.edge_fn
    )

    edge_jaccard = (
        result.edge_tp / denom
        if denom > 0
        else np.nan
    )

    boundary_records.append({
        "distance_um": distance_um,
        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,
        "edge_jaccard": edge_jaccard,
    })

boundary_df = pd.DataFrame(
    boundary_records
)

boundary_df

官方实际行为是：d ≤ 7.0 μm ⇒ match


In [ ]:
def make_two_node_track_voxel_shift(
    dz=0.0,
    dy=0.0,
    dx=0.0,
):
    graph = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        graph.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    n0 = graph.add_node(
        attrs={
            "t": 0,
            "z": 20.0 + dz,
            "y": 100.0 + dy,
            "x": 100.0 + dx,
        }
    )

    n1 = graph.add_node(
        attrs={
            "t": 1,
            "z": 20.0 + dz,
            "y": 100.0 + dy,
            "x": 100.0 + dx,
        }
    )

    graph.add_edge(
        n0,
        n1,
        {},
    )

    return graph

In [ ]:
anisotropy_cases = [
    {
        "case": "5 x-voxels",
        "dz": 0,
        "dy": 0,
        "dx": 5,
    },
    {
        "case": "5 y-voxels",
        "dz": 0,
        "dy": 5,
        "dx": 0,
    },
    {
        "case": "5 z-voxels",
        "dz": 5,
        "dy": 0,
        "dx": 0,
    },
    {
        "case": "4 z-voxels",
        "dz": 4,
        "dy": 0,
        "dx": 0,
    },
]

anisotropy_records = []

for case in anisotropy_cases:

    gt_graph = make_two_node_track_voxel_shift()

    pred_graph = make_two_node_track_voxel_shift(
        dz=case["dz"],
        dy=case["dy"],
        dx=case["dx"],
    )

    physical_distance_um = np.sqrt(
        (case["dz"] * PHYSICAL_SCALE[0]) ** 2
        +
        (case["dy"] * PHYSICAL_SCALE[1]) ** 2
        +
        (case["dx"] * PHYSICAL_SCALE[2]) ** 2
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    anisotropy_records.append({
        "case": case["case"],
        "physical_distance_um":
            physical_distance_um,
        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,
    })

anisotropy_df = pd.DataFrame(
    anisotropy_records
)

anisotropy_df

anisotropic scale： 

d = sqrt[(1.625Δz)^2 +(0.40625Δy)^2+ (0.40625Δx)^2]

d ≤ 7.0 μm ⇒ match


In [ ]:
def make_asymmetric_track(
    source_shift_um=0.0,
    target_shift_um=0.0,
):
    """
    One edge:
        source(t=0) -> target(t=1)

    source and target can be shifted independently
    along x in physical microns.
    """

    graph = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        graph.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    source_dx = (
        source_shift_um
        / PHYSICAL_SCALE[2]
    )

    target_dx = (
        target_shift_um
        / PHYSICAL_SCALE[2]
    )

    source = graph.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0 + source_dx,
        }
    )

    target = graph.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0 + target_dx,
        }
    )

    graph.add_edge(
        source,
        target,
        {},
    )

    return graph

In [ ]:
sparse_cases = [
    {
        "case": "both matched",
        "source_shift_um": 0.0,
        "target_shift_um": 0.0,
    },
    {
        "case": "source matched, target unmatched",
        "source_shift_um": 0.0,
        "target_shift_um": 8.0,
    },
    {
        "case": "source unmatched, target matched",
        "source_shift_um": 8.0,
        "target_shift_um": 0.0,
    },
    {
        "case": "both unmatched",
        "source_shift_um": 8.0,
        "target_shift_um": 8.0,
    },
]

sparse_edge_records = []

for case in sparse_cases:

    gt_graph = make_asymmetric_track(
        0.0,
        0.0,
    )

    pred_graph = make_asymmetric_track(
        source_shift_um=
            case["source_shift_um"],
        target_shift_um=
            case["target_shift_um"],
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    sparse_edge_records.append({
        "case": case["case"],
        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,
    })

sparse_edge_df = pd.DataFrame(
    sparse_edge_records
)

sparse_edge_df

预测 edge 两端都完全 unmatched 时，这条 edge 不产生 edge FP​

In [ ]:
def make_track_with_optional_ghost(
    add_ghost=False,
):

    graph = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        graph.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    # Correct track
    a = graph.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    b = graph.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    graph.add_edge(a, b, {})

    if add_ghost:

        # Far-away unmatched track
        g0 = graph.add_node(
            attrs={
                "t": 0,
                "z": 50.0,
                "y": 200.0,
                "x": 200.0,
            }
        )

        g1 = graph.add_node(
            attrs={
                "t": 1,
                "z": 50.0,
                "y": 200.0,
                "x": 200.0,
            }
        )

        graph.add_edge(
            g0,
            g1,
            {},
        )

    return graph

In [ ]:
from tracking_cellmot.metrics import (
    evaluate,
    per_sample_metrics,
)

ghost_records = []

for add_ghost in [
    False,
    True,
]:

    gt_graph = make_track_with_optional_ghost(
        add_ghost=False
    )

    pred_graph = make_track_with_optional_ghost(
        add_ghost=add_ghost
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    edge_denom = (
        result.edge_tp
        + result.edge_fp
        + result.edge_fn
    )

    raw_edge_jaccard = (
        result.edge_tp / edge_denom
    )

    # Synthetic ground truth has exactly 2 intended nodes.
    metrics = per_sample_metrics(
        result,
        n_total=2,
        node_recall=1.0,
    )

    ghost_records.append({
        "prediction":
            "correct + ghost"
            if add_ghost
            else "correct only",

        "num_pred_nodes":
            result.num_pred_nodes,

        "edge_tp":
            result.edge_tp,

        "edge_fp":
            result.edge_fp,

        "edge_fn":
            result.edge_fn,

        "raw_edge_jaccard":
            raw_edge_jaccard,

        "total_node_ratio":
            metrics[
                "total_node_ratio"
            ],

        "adjusted_edge_jaccard":
            metrics[
                "adj_edge_jaccard"
            ],
    })

ghost_df = pd.DataFrame(
    ghost_records
)

ghost_df

In [ ]:
def make_gt_with_isolated_nodes():

    graph = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        graph.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    # One tracked edge
    a = graph.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    b = graph.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    graph.add_edge(a, b, {})

    # Two isolated GT nodes
    graph.add_node(
        attrs={
            "t": 0,
            "z": 40.0,
            "y": 50.0,
            "x": 50.0,
        }
    )

    graph.add_node(
        attrs={
            "t": 1,
            "z": 45.0,
            "y": 200.0,
            "x": 200.0,
        }
    )

    return graph


def make_edge_only_prediction():

    graph = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        graph.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    a = graph.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    b = graph.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    graph.add_edge(a, b, {})

    return graph

In [ ]:
from tracking_cellmot.metrics import (
    evaluate,
    node_recall,
    per_sample_metrics,
)

gt_graph = make_gt_with_isolated_nodes()
pred_graph = make_edge_only_prediction()

result = evaluate(
    pred_graph,
    gt_graph,
    scale=PHYSICAL_SCALE,
    max_distance=7.0,
)

recall = node_recall(
    pred_graph,
    gt_graph,
)

metrics = per_sample_metrics(
    result,
    n_total=gt_graph.num_nodes(),
    node_recall=recall,
)

print(
    "GT nodes:",
    gt_graph.num_nodes(),
)

print(
    "Pred nodes:",
    result.num_pred_nodes,
)

print(
    "Edge TP / FP / FN:",
    result.edge_tp,
    result.edge_fp,
    result.edge_fn,
)

print(
    "Node recall:",
    recall,
)

print(
    "Raw edge Jaccard:",
    metrics["edge_jaccard"],
)

print(
    "Total node ratio:",
    metrics["total_node_ratio"],
)

print(
    "Adjusted edge Jaccard:",
    metrics["adj_edge_jaccard"],
)

Adjusted Edge Jaccard 可以超过 1​

In [ ]:
import tracksdata as td

geff_path = (
    TRAIN_DIR
    / f"{sanity_stem}.geff"
)

loaded = td.graph.IndexedRXGraph.from_geff(
    geff_path
)

print(
    "Return type:",
    type(loaded),
)

if isinstance(loaded, tuple):
    print(
        "Tuple length:",
        len(loaded),
    )

    for i, item in enumerate(loaded):
        print(
            f"\n--- item {i} ---"
        )
        print(
            "type:",
            type(item),
        )
        print(item)
else:
    print(loaded)

estimated total nodes 大约是 sparse GT nodes 的 74.5 倍
	​


平均每帧则大约是：

Sparse annotations ≈ 7.9 nodes/frame
Estimated real nodes ≈ 586.7 nodes/frame

In [ ]:
node_count_records = []

train_geffs = sorted(
    TRAIN_DIR.glob("*.geff")
)

for geff_path in train_geffs:

    stem = geff_path.stem
    prefix = stem.split("_")[0]

    loaded = td.graph.IndexedRXGraph.from_geff(
        geff_path
    )

    if isinstance(loaded, tuple):
        graph, metadata = loaded
    else:
        raise RuntimeError(
            f"No GEFF metadata returned for {stem}"
        )

    estimated_nodes = (
        metadata.extra.get(
            "estimated_number_of_nodes"
        )
        if metadata.extra
        else None
    )

    # Number of time points from metadata
    t_axis = next(
        axis
        for axis in metadata.axes
        if axis.name == "t"
    )

    n_frames = int(
        t_axis.max
        - t_axis.min
        + 1
    )

    sparse_gt_nodes = (
        graph.num_nodes()
    )

    node_count_records.append({
        "dataset": stem,
        "prefix": prefix,

        "n_frames":
            n_frames,

        "sparse_gt_nodes":
            sparse_gt_nodes,

        "estimated_nodes":
            estimated_nodes,

        "sparse_fraction":
            (
                sparse_gt_nodes
                / estimated_nodes
            ),

        "estimated_per_frame":
            (
                estimated_nodes
                / n_frames
            ),

        "sparse_gt_per_frame":
            (
                sparse_gt_nodes
                / n_frames
            ),

        "estimated_to_sparse_ratio":
            (
                estimated_nodes
                / sparse_gt_nodes
            ),
    })

node_count_df = pd.DataFrame(
    node_count_records
)

node_count_df.head()

In [ ]:
node_count_summary = (
    node_count_df
    .groupby("prefix")
    .agg(
        n_datasets=(
            "dataset",
            "size",
        ),

        estimated_nodes_mean=(
            "estimated_nodes",
            "mean",
        ),

        estimated_nodes_median=(
            "estimated_nodes",
            "median",
        ),

        estimated_nodes_min=(
            "estimated_nodes",
            "min",
        ),

        estimated_nodes_max=(
            "estimated_nodes",
            "max",
        ),

        estimated_per_frame_median=(
            "estimated_per_frame",
            "median",
        ),

        sparse_gt_per_frame_median=(
            "sparse_gt_per_frame",
            "median",
        ),

        sparse_fraction_median=(
            "sparse_fraction",
            "median",
        ),

        estimated_to_sparse_ratio_median=(
            "estimated_to_sparse_ratio",
            "median",
        ),
    )
)

node_count_summary

In [ ]:
node_count_df.groupby(
    "prefix"
)[
    [
        "estimated_per_frame",
        "sparse_fraction",
        "estimated_to_sparse_ratio",
    ]
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
    ]
)

44b6 的 supervision 比 6bba 稀疏大约一个数量级​a

44b6: 500/frame 是正常的量级

6bba: 100/frame 是正常量级

estimated_number_of_nodes 本身变化非常大


per-dataset cell-density / node-count estimation

In [ ]:
node_ratio_records = []

prediction_ratios = [
    0.25,
    0.50,
    0.75,
    0.90,
    1.00,
    1.10,
    1.25,
    1.50,
    2.00,
    3.00,
    5.00,
    10.00,
]

raw_jaccards = [
    0.25,
    0.50,
    0.75,
    1.00,
]

for raw_j in raw_jaccards:

    for pred_ratio in prediction_ratios:

        total_node_ratio = (
            pred_ratio - 1.0
        )

        adjusted = max(
            0.0,
            raw_j
            * (
                1
                - 0.1
                * total_node_ratio
            )
        )

        node_ratio_records.append({
            "raw_edge_jaccard":
                raw_j,

            "pred_to_total_ratio":
                pred_ratio,

            "node_count_error_pct":
                100 * (
                    pred_ratio - 1
                ),

            "adjusted_edge_jaccard":
                adjusted,

            "adjustment_factor":
                (
                    adjusted / raw_j
                    if raw_j > 0
                    else np.nan
                ),
        })

node_penalty_df = pd.DataFrame(
    node_ratio_records
)


In [ ]:
node_penalty_pivot = (
    node_penalty_df
    .pivot(
        index="pred_to_total_ratio",
        columns="raw_edge_jaccard",
        values="adjustment_factor",
    )
)

node_penalty_pivot

node-count penalty 本身不算强

In [ ]:
def make_gt_division():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    parent = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    d1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 90.0,
            "x": 100.0,
        }
    )

    d2 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 110.0,
            "x": 100.0,
        }
    )

    g.add_edge(parent, d1, {})
    g.add_edge(parent, d2, {})

    return g


def make_pred_division(case):

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    parent = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    d1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 90.0,
            "x": 100.0,
        }
    )

    d2 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 110.0,
            "x": 100.0,
        }
    )

    if case == "correct":

        g.add_edge(parent, d1, {})
        g.add_edge(parent, d2, {})

    elif case == "missing_second_edge":

        # All three nodes are correctly detected,
        # but only one daughter is linked.
        g.add_edge(parent, d1, {})

    elif case == "wrong_second_branch":

        # Correct second daughter still exists,
        # but fork points to a distant ghost instead.
        ghost = g.add_node(
            attrs={
                "t": 1,
                "z": 50.0,
                "y": 220.0,
                "x": 220.0,
            }
        )

        g.add_edge(parent, d1, {})
        g.add_edge(parent, ghost, {})

    else:
        raise ValueError(case)

    return g

In [ ]:
division_cases = [
    "correct",
    "missing_second_edge",
    "wrong_second_branch",
]

division_records = []

for case in division_cases:

    gt_graph = make_gt_division()
    pred_graph = make_pred_division(case)

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    division_records.append({
        "case": case,

        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,

        "division_tp":
            result.division_tp,

        "division_fp":
            result.division_fp,

        "division_fn":
            result.division_fn,
    })

division_basic_df = pd.DataFrame(
    division_records
)

division_basic_df

correct                    → division TP=1

missing_second_edge        → division FN=1

wrong_second_branch        → division FP=1 + FN=1

In [ ]:
def make_three_child_prediction(edge_order):

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    parent = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    d1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 90.0,
            "x": 100.0,
        }
    )

    d2 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 110.0,
            "x": 100.0,
        }
    )

    ghost = g.add_node(
        attrs={
            "t": 1,
            "z": 50.0,
            "y": 220.0,
            "x": 220.0,
        }
    )

    targets = {
        "d1": d1,
        "d2": d2,
        "ghost": ghost,
    }

    for name in edge_order:
        g.add_edge(
            parent,
            targets[name],
            {},
        )

    return g

In [ ]:
three_child_cases = [
    {
        "case": "d1, d2, ghost",
        "order": ["d1", "d2", "ghost"],
    },
    {
        "case": "d1, ghost, d2",
        "order": ["d1", "ghost", "d2"],
    },
    {
        "case": "ghost, d1, d2",
        "order": ["ghost", "d1", "d2"],
    },
]

three_child_records = []

for case in three_child_cases:

    gt_graph = make_gt_division()

    pred_graph = make_three_child_prediction(
        case["order"]
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    edge_denom = (
        result.edge_tp
        + result.edge_fp
        + result.edge_fn
    )

    edge_jaccard = (
        result.edge_tp / edge_denom
        if edge_denom > 0
        else np.nan
    )

    division_denom = (
        result.division_tp
        + result.division_fp
        + result.division_fn
    )

    division_jaccard = (
        result.division_tp / division_denom
        if division_denom > 0
        else np.nan
    )

    three_child_records.append({
        "case": case["case"],
        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,
        "edge_jaccard": edge_jaccard,
        "division_tp": result.division_tp,
        "division_fp": result.division_fp,
        "division_fn": result.division_fn,
        "division_jaccard": division_jaccard,
    })

three_child_df = pd.DataFrame(
    three_child_records
)

three_child_df

最终提交图必须主动保证 out-degree≤2​

In [ ]:
def make_three_frame_gt():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    nodes = []

    for t in [0, 1, 2]:
        node = g.add_node(
            attrs={
                "t": t,
                "z": 20.0,
                "y": 100.0,
                "x": 100.0,
            }
        )
        nodes.append(node)

    g.add_edge(nodes[0], nodes[1], {})
    g.add_edge(nodes[1], nodes[2], {})

    return g

In [ ]:
def make_temporal_edge_prediction(case):

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    nodes = []

    for t in [0, 1, 2]:
        node = g.add_node(
            attrs={
                "t": t,
                "z": 20.0,
                "y": 100.0,
                "x": 100.0,
            }
        )
        nodes.append(node)

    if case == "correct":
        g.add_edge(nodes[0], nodes[1], {})
        g.add_edge(nodes[1], nodes[2], {})

    elif case == "gap_only":
        # t=0 → t=2
        g.add_edge(nodes[0], nodes[2], {})

    elif case == "one_correct_plus_gap":
        g.add_edge(nodes[0], nodes[1], {})
        g.add_edge(nodes[0], nodes[2], {})

    elif case == "backward":
        # t=1 → t=0
        g.add_edge(nodes[1], nodes[0], {})

    else:
        raise ValueError(case)

    return g

In [ ]:
temporal_cases = [
    "correct",
    "gap_only",
    "one_correct_plus_gap",
    "backward",
]

temporal_records = []

for case in temporal_cases:

    gt_graph = make_three_frame_gt()

    pred_graph = (
        make_temporal_edge_prediction(case)
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    denom = (
        result.edge_tp
        + result.edge_fp
        + result.edge_fn
    )

    temporal_records.append({
        "case": case,
        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,
        "edge_jaccard":
            result.edge_tp / denom
            if denom > 0
            else np.nan,
    })

temporal_edge_df = pd.DataFrame(
    temporal_records
)

temporal_edge_df

只有 Δt=1 的 edge 会进入 Edge metric

In [ ]:
def make_gt_chain():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    n0 = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    n1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    n2 = g.add_node(
        attrs={
            "t": 2,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    g.add_edge(n0, n1, {})
    g.add_edge(n1, n2, {})

    return g

In [ ]:
def make_spurious_division_prediction(case):

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    n0 = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    n1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    n2 = g.add_node(
        attrs={
            "t": 2,
            "z": 20.0,
            "y": 100.0,
            "x": 100.0,
        }
    )

    g.add_edge(n0, n1, {})
    g.add_edge(n1, n2, {})

    if case == "correct_chain":
        pass

    elif case == "fork_at_interior":

        # n1 is an annotated interior GT node.
        # GT says it continues normally to n2.
        ghost = g.add_node(
            attrs={
                "t": 2,
                "z": 40.0,
                "y": 200.0,
                "x": 200.0,
            }
        )

        g.add_edge(n1, ghost, {})

    elif case == "fork_at_terminal":

        # n2 matches the terminal annotated GT node.
        # There is no GT after t=2.
        g1 = g.add_node(
            attrs={
                "t": 3,
                "z": 40.0,
                "y": 190.0,
                "x": 190.0,
            }
        )

        g2 = g.add_node(
            attrs={
                "t": 3,
                "z": 45.0,
                "y": 210.0,
                "x": 210.0,
            }
        )

        g.add_edge(n2, g1, {})
        g.add_edge(n2, g2, {})

    elif case == "unmatched_ghost_fork":

        parent = g.add_node(
            attrs={
                "t": 1,
                "z": 55.0,
                "y": 220.0,
                "x": 220.0,
            }
        )

        c1 = g.add_node(
            attrs={
                "t": 2,
                "z": 55.0,
                "y": 210.0,
                "x": 220.0,
            }
        )

        c2 = g.add_node(
            attrs={
                "t": 2,
                "z": 55.0,
                "y": 230.0,
                "x": 220.0,
            }
        )

        g.add_edge(parent, c1, {})
        g.add_edge(parent, c2, {})

    else:
        raise ValueError(case)

    return g

In [ ]:
division_sparse_cases = [
    "correct_chain",
    "fork_at_interior",
    "fork_at_terminal",
    "unmatched_ghost_fork",
]

division_sparse_records = []

for case in division_sparse_cases:

    gt_graph = make_gt_chain()

    pred_graph = (
        make_spurious_division_prediction(
            case
        )
    )

    result = evaluate(
        pred_graph,
        gt_graph,
        scale=PHYSICAL_SCALE,
        max_distance=7.0,
    )

    division_sparse_records.append({
        "case": case,

        "edge_tp": result.edge_tp,
        "edge_fp": result.edge_fp,
        "edge_fn": result.edge_fn,

        "division_tp":
            result.division_tp,

        "division_fp":
            result.division_fp,

        "division_fn":
            result.division_fn,
    })

division_sparse_df = pd.DataFrame(
    division_sparse_records
)

division_sparse_df

Division FP 只在 GT 足以判断该 fork 错误时才处罚
	​


In [ ]:
def make_two_component_gt():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    # Component A
    a0 = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    a1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    g.add_edge(a0, a1, {})

    # Completely separate component B
    b0 = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    b1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    g.add_edge(b0, b1, {})

    return g

In [ ]:
def make_cross_component_prediction():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    # Matches GT A0
    parent = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    # Matches GT A1
    child_a = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    # Matches GT B1
    child_b = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    # False fork:
    # A0 -> A1
    #    -> B1
    g.add_edge(
        parent,
        child_a,
        {},
    )

    g.add_edge(
        parent,
        child_b,
        {},
    )

    return g

In [ ]:
gt_graph = make_two_component_gt()

pred_graph = (
    make_cross_component_prediction()
)

result = evaluate(
    pred_graph,
    gt_graph,
    scale=PHYSICAL_SCALE,
    max_distance=7.0,
)

print(
    "Edge TP / FP / FN:",
    result.edge_tp,
    result.edge_fp,
    result.edge_fn,
)

print(
    "Division TP / FP / FN:",
    result.division_tp,
    result.division_fp,
    result.division_fn,
)

官方实现确实会检查预测 fork 不同 branch 的 GT component evidence

In [ ]:
def make_two_component_long_gt():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    # Component A
    a0 = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    a1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    a2 = g.add_node(
        attrs={
            "t": 2,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    g.add_edge(a0, a1, {})
    g.add_edge(a1, a2, {})

    # Component B
    b0 = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    b1 = g.add_node(
        attrs={
            "t": 1,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    b2 = g.add_node(
        attrs={
            "t": 2,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    g.add_edge(b0, b1, {})
    g.add_edge(b1, b2, {})

    return g

In [ ]:
def make_grandchild_cross_component_prediction():

    g = td.graph.InMemoryGraph()

    for attr in ["z", "y", "x"]:
        g.add_node_attr_key(
            attr,
            pl.Float64,
            0.0,
        )

    # Matches GT A0
    parent = g.add_node(
        attrs={
            "t": 0,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    # Both predicted children are deliberately
    # far from all GT nodes at t=1.
    child_a = g.add_node(
        attrs={
            "t": 1,
            "z": 50.0,
            "y": 30.0,
            "x": 30.0,
        }
    )

    child_b = g.add_node(
        attrs={
            "t": 1,
            "z": 50.0,
            "y": 220.0,
            "x": 220.0,
        }
    )

    # Grandchild matches GT A2
    grandchild_a = g.add_node(
        attrs={
            "t": 2,
            "z": 20.0,
            "y": 80.0,
            "x": 100.0,
        }
    )

    # Grandchild matches GT B2
    grandchild_b = g.add_node(
        attrs={
            "t": 2,
            "z": 20.0,
            "y": 120.0,
            "x": 100.0,
        }
    )

    g.add_edge(parent, child_a, {})
    g.add_edge(parent, child_b, {})

    g.add_edge(child_a, grandchild_a, {})
    g.add_edge(child_b, grandchild_b, {})

    return g

In [ ]:
gt_graph = make_two_component_long_gt()

pred_graph = (
    make_grandchild_cross_component_prediction()
)

result = evaluate(
    pred_graph,
    gt_graph,
    scale=PHYSICAL_SCALE,
    max_distance=7.0,
)

print(
    "Edge TP / FP / FN:",
    result.edge_tp,
    result.edge_fp,
    result.edge_fn,
)

print(
    "Division TP / FP / FN:",
    result.division_tp,
    result.division_fp,
    result.division_fn,
)

直接 child 无匹配时，可以使用 grandchild 的 component evidence 作为 fallback